In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
import json
import pprint

print("Everything is working!")

Everything is working!


In [2]:
DATA_PATH = Path("../data/candidates.jsonl")

print(DATA_PATH)
print(DATA_PATH.exists())

..\data\candidates.jsonl
True


In [3]:
candidates = []

with open(DATA_PATH, "r", encoding="utf-8") as file:
    for line in tqdm(file):
        candidates.append(json.loads(line))

print(len(candidates))

100000it [00:04, 20944.66it/s]

100000


In [4]:
print(candidates[0].keys())

dict_keys(['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals'])


In [5]:
pp = pprint.PrettyPrinter(indent=2, width=120)
pp.pprint(candidates[0])

{ 'candidate_id': 'CAND_0000001',
  'career_history': [ { 'company': 'Mindtree',
                        'company_size': '10001+',
                        'description': 'Implemented streaming data pipelines on Kafka and Spark Streaming for a '
                                       'real-time user-activity processing platform. Designed the schema-registry '
                                       'integration, the watermark/state management approach, and the deduplication '
                                       'logic for late-arriving events. Worked closely with the data science team to '
                                       'make sure feature pipelines aligned with what their models needed. Most of my '
                                       'career has been data engineering, with some adjacent ML exposure.',
                        'duration_months': 27,
                        'end_date': None,
                        'industry': 'IT Services',
                        'is_curren

In [6]:
print(candidates[0]["candidate_id"])

CAND_0000001


In [7]:
print(candidates[0]["profile"].keys())

dict_keys(['anonymized_name', 'headline', 'summary', 'location', 'country', 'years_of_experience', 'current_title', 'current_company', 'current_company_size', 'current_industry'])


In [8]:
print(candidates[0]["redrob_signals"].keys())

dict_keys(['profile_completeness_score', 'signup_date', 'last_active_date', 'open_to_work_flag', 'profile_views_received_30d', 'applications_submitted_30d', 'recruiter_response_rate', 'avg_response_time_hours', 'skill_assessment_scores', 'connection_count', 'endorsements_received', 'notice_period_days', 'expected_salary_range_inr_lpa', 'preferred_work_mode', 'willing_to_relocate', 'github_activity_score', 'search_appearance_30d', 'saved_by_recruiters_30d', 'interview_completion_rate', 'offer_acceptance_rate', 'verified_email', 'verified_phone', 'linkedin_connected'])


# 3. Candidate Consistency Analysis

# Candidate Dataset Exploration

## 1. Basic Dataset Overview

In [10]:
from pathlib import Path
import json
from tqdm import tqdm

DATA_PATH = Path("../data/candidates.jsonl")

candidates = []

with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in tqdm(f):
        candidates.append(json.loads(line))

print("Total candidates:", len(candidates))

sample = candidates[0]

print("\nTop level keys:")
print(sample.keys())

print("\nProfile keys:")
print(sample["profile"].keys())

print("\nBehavior keys:")
print(sample["redrob_signals"].keys())

100000it [00:05, 18659.70it/s]

Total candidates: 100000

Top level keys:
dict_keys(['candidate_id', 'profile', 'career_history', 'education', 'skills', 'certifications', 'languages', 'redrob_signals'])

Profile keys:
dict_keys(['anonymized_name', 'headline', 'summary', 'location', 'country', 'years_of_experience', 'current_title', 'current_company', 'current_company_size', 'current_industry'])

Behavior keys:
dict_keys(['profile_completeness_score', 'signup_date', 'last_active_date', 'open_to_work_flag', 'profile_views_received_30d', 'applications_submitted_30d', 'recruiter_response_rate', 'avg_response_time_hours', 'skill_assessment_scores', 'connection_count', 'endorsements_received', 'notice_period_days', 'expected_salary_range_inr_lpa', 'preferred_work_mode', 'willing_to_relocate', 'github_activity_score', 'search_appearance_30d', 'saved_by_recruiters_30d', 'interview_completion_rate', 'offer_acceptance_rate', 'verified_email', 'verified_phone', 'linkedin_connected'])


## Candidate Profile Overview

In [11]:
import pandas as pd

profile_df = pd.DataFrame([
    {
        "candidate_id": c["candidate_id"],
        "title": c["profile"]["current_title"],
        "industry": c["profile"]["current_industry"],
        "country": c["profile"]["country"],
        "experience": c["profile"]["years_of_experience"],
        "skills_count": len(c["skills"]),
        "career_count": len(c["career_history"]),
        "certifications_count": len(c["certifications"])
    }
    for c in candidates
])

print(profile_df.shape)

display(profile_df.head())

print("\nExperience statistics:")
print(profile_df["experience"].describe())

print("\nTop 20 titles:")
print(profile_df["title"].value_counts().head(20))

print("\nTop 20 industries:")
print(profile_df["industry"].value_counts().head(20))

(100000, 8)


,candidate_id,title,industry,country,experience,skills_count,career_count,certifications_count
0,CAND_0000001,Backend Engineer,IT Services,Canada,6.9,17,2,0
1,CAND_0000002,Operations Manager,IT Services,India,12.5,9,4,0
2,CAND_0000003,Customer Support,IT Services,USA,1.1,6,1,0
3,CAND_0000004,Marketing Manager,Paper Products,Australia,3.8,10,3,2
4,CAND_0000005,Accountant,Manufacturing,India,11.0,6,4,0



Experience statistics:
count    100000.000000
mean          7.166319
std           3.824551
min           1.000000
25%           3.900000
50%           6.800000
75%           9.900000
max          16.900000
Name: experience, dtype: float64

Top 20 titles:
title
Business Analyst        5833
HR Manager              5830
Mechanical Engineer     5791
Accountant              5764
Project Manager         5754
Customer Support        5750
Operations Manager      5744
Content Writer          5727
Sales Executive         5713
Civil Engineer          5702
Graphic Designer        5689
Marketing Manager       5524
Software Engineer       3450
Full Stack Developer    2873
Cloud Engineer          2836
Java Developer          2809
.NET Developer          2788
DevOps Engineer         2787
Mobile Developer        2757
Frontend Engineer       2738
Name: count, dtype: int64

Top 20 industries:
industry
IT Services          29881
Software             22417
Manufacturing        22305
Conglomerate         

## Skills Analysis

In [12]:
from collections import Counter

skill_counter = Counter()
skill_counts = []

for c in candidates:
    skills = c["skills"]
    skill_counts.append(len(skills))

    for skill in skills:
        skill_counter[skill["name"]] += 1

print("Skill count statistics:")
print(pd.Series(skill_counts).describe())

print("\nTop 30 skills:")
top_skills = pd.DataFrame(
    skill_counter.most_common(30),
    columns=["skill", "count"]
)

display(top_skills)

Skill count statistics:
count    100000.00000
mean          9.60302
std           3.31163
min           5.00000
25%           7.00000
50%           9.00000
75%          11.00000
max          23.00000
dtype: float64

Top 30 skills:


,skill,count
0,HTML,12246
1,Databricks,12244
2,Redux,12222
3,Terraform,12187
4,Angular,12173
5,Figma,12157
6,Salesforce CRM,12157
7,Vue.js,12142
8,Sales,12138
9,Accounting,12136


## Honeypot Analysis - Salary and Date Consistency

In [13]:
salary_error = 0
date_error = 0

for c in candidates:
    signals = c["redrob_signals"]

    salary = signals["expected_salary_range_inr_lpa"]
    if salary["min"] > salary["max"]:
        salary_error += 1

    if signals["signup_date"] > signals["last_active_date"]:
        date_error += 1

print("Salary inconsistencies:", salary_error)
print("Date inconsistencies:", date_error)

Salary inconsistencies: 18865
Date inconsistencies: 7496


## Honeypot Analysis - Experience Consistency

In [14]:
experience_errors = 0

for c in candidates:
    claimed = c["profile"]["years_of_experience"]

    actual = sum(
        job["duration_months"]
        for job in c["career_history"]
    ) / 12

    if abs(claimed - actual) > 2:
        experience_errors += 1

print("Experience inconsistencies:", experience_errors)
print("Percentage:", round(experience_errors / len(candidates) * 100, 2), "%")

Experience inconsistencies: 48
Percentage: 0.05 %


In [15]:
for c in candidates:
    claimed = c["profile"]["years_of_experience"]

    actual = sum(
        job["duration_months"]
        for job in c["career_history"]
    ) / 12

    if abs(claimed - actual) > 8:
        print("\nCandidate:", c["candidate_id"])
        print("Claimed:", claimed)
        print("Actual:", round(actual,2))

        for job in c["career_history"]:
            print(
                job["title"],
                "-",
                job["duration_months"],
                "months"
            )

        break


Candidate: CAND_0003430
Claimed: 13.7
Actual: 0.92
Business Analyst - 11 months


## Experience Mismatch Investigation

In [16]:
bad_candidates = []

for c in candidates:
    claimed = c["profile"]["years_of_experience"]

    actual = sum(
        job["duration_months"]
        for job in c["career_history"]
    ) / 12

    diff = abs(claimed - actual)

    if diff > 5:
        bad_candidates.append({
            "candidate_id": c["candidate_id"],
            "claimed": claimed,
            "actual": round(actual,2),
            "difference": round(diff,2),
            "jobs": len(c["career_history"])
        })

bad_df = pd.DataFrame(bad_candidates)

print("Total severe mismatches:", len(bad_df))
display(bad_df.head(20))

Total severe mismatches: 39


,candidate_id,claimed,actual,difference,jobs
0,CAND_0003430,13.7,0.92,12.78,1
1,CAND_0005291,12.8,0.92,11.88,1
2,CAND_0007353,9.9,20.92,11.02,4
3,CAND_0007413,13.3,1.33,11.97,1
4,CAND_0008960,10.3,22.58,12.28,4
5,CAND_0010294,8.0,18.33,10.33,3
6,CAND_0010770,15.2,7.17,8.03,4
7,CAND_0013536,14.1,4.67,9.43,2
8,CAND_0018515,8.5,17.58,9.08,3
9,CAND_0024752,14.9,0.67,14.23,1


## Current Job Consistency

In [17]:
current_job_errors = 0

for c in candidates:
    current_jobs = sum(
        1 for job in c["career_history"]
        if job["is_current"]
    )

    if current_jobs != 1:
        current_job_errors += 1

print("Current job inconsistencies:", current_job_errors)
print(
    "Percentage:",
    round(current_job_errors/len(candidates)*100,2),
    "%"
)

Current job inconsistencies: 0
Percentage: 0.0 %


In [18]:
examples = []

for c in candidates:
    current_jobs = [
        job for job in c["career_history"]
        if job["is_current"]
    ]

    if len(current_jobs) != 1:
        examples.append({
            "candidate_id": c["candidate_id"],
            "current_jobs": len(current_jobs),
            "career_length": len(c["career_history"])
        })

bad_current = pd.DataFrame(examples)

display(bad_current.head(10))

""


## Salary Range Analysis

In [19]:
salary_diffs = []

for c in candidates:
    salary = c["redrob_signals"]["expected_salary_range_inr_lpa"]

    salary_diffs.append({
        "candidate_id": c["candidate_id"],
        "min_salary": salary["min"],
        "max_salary": salary["max"],
        "difference": salary["max"] - salary["min"]
    })

salary_df = pd.DataFrame(salary_diffs)

print(salary_df.describe())

print("\nNegative salary ranges:")
print((salary_df["difference"] < 0).sum())

print("\nVery large salary ranges (>100 LPA):")
print((salary_df["difference"] > 100).sum())

          min_salary     max_salary     difference
count  100000.000000  100000.000000  100000.000000
mean       12.172076      19.842671       7.670595
std         5.564816       8.318941       8.163889
min         3.000000       6.000000     -12.000000
25%         7.800000      13.500000       1.700000
50%        11.900000      19.400000       7.600000
75%        15.800000      25.200000      13.500000
max        49.700000      74.500000      41.900000

Negative salary ranges:
18865

Very large salary ranges (>100 LPA):
0


In [20]:
salary_df[salary_df["difference"] < 0].head(10)

,candidate_id,min_salary,max_salary,difference
8,CAND_0000009,16.0,7.3,-8.7
10,CAND_0000011,15.5,13.9,-1.6
11,CAND_0000012,14.8,7.9,-6.9
12,CAND_0000013,11.6,8.1,-3.5
14,CAND_0000015,21.8,18.9,-2.9
16,CAND_0000017,13.8,8.4,-5.4
18,CAND_0000019,12.5,7.7,-4.8
21,CAND_0000022,12.3,8.5,-3.8
25,CAND_0000026,17.1,8.0,-9.1
29,CAND_0000030,14.7,14.2,-0.5


## Behavioral Signal Analysis

In [21]:
behavior = []

for c in candidates:
    s = c["redrob_signals"]

    behavior.append({
        "github": s["github_activity_score"],
        "response_rate": s["recruiter_response_rate"],
        "interview_rate": s["interview_completion_rate"],
        "offer_rate": s["offer_acceptance_rate"],
        "profile_score": s["profile_completeness_score"],
        "connections": s["connection_count"],
        "open_to_work": s["open_to_work_flag"]
    })

behavior_df = pd.DataFrame(behavior)

print(behavior_df.describe())

print("\nOpen to work:")
print(behavior_df["open_to_work"].value_counts())

              github  response_rate  interview_rate     offer_rate  \
count  100000.000000  100000.000000   100000.000000  100000.000000   
mean        9.619230       0.436574        0.619510      -0.403604   
std        17.761394       0.214122        0.170662       0.732439   
min        -1.000000       0.020000        0.300000      -1.000000   
25%        -1.000000       0.250000        0.480000      -1.000000   
50%        -1.000000       0.440000        0.620000      -1.000000   
75%        16.700000       0.620000        0.760000       0.400000   
max        96.900000       0.950000        1.000000       0.930000   

       profile_score    connections  
count  100000.000000  100000.000000  
mean       56.758180     345.664890  
std        17.274069     208.145694  
min        25.000000      10.000000  
25%        42.200000     174.000000  
50%        56.800000     335.000000  
75%        71.600000     497.000000  
max        99.900000    1898.000000  

Open to work:
open_to_work

# temp code

In [23]:
import sys
sys.path.append("../src")

In [26]:
from honeypot import detect_honeypots

count = 0

for c in candidates:
    result = detect_honeypots(c)

    if result["honeypot_score"] > 0:
        count += 1

print("Candidates with honeypots:", count)
print("Percentage:", round(count/len(candidates)*100, 2), "%")

Candidates with honeypots: 24925
Percentage: 24.93 %
